# MoE MarioGPT - Colab Training
**使用前請先確認：Runtime → Change runtime type → A100 GPU**

### 步驟：
1. 把整個 `mario-gpt-moe-mlp-2DROPE` 資料夾上傳到你的 Google Drive
2. 依序執行所有 Cell

In [ ]:
# Cell 1 - 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 - 設定路徑（依照你在 Drive 裡的實際位置修改）
PROJECT_DIR = "/content/drive/MyDrive/mario-gpt-moe-mlp-2DROPE"
OUTPUT_DIR  = "/content/drive/MyDrive/mario-gpt-output"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Project : {PROJECT_DIR}")
print(f"Output  : {OUTPUT_DIR}")

In [ ]:
# Cell 3 - 安裝依賴套件（只需執行一次）
import subprocess, sys

%pip install -q transformers>=4.36.0 accelerate>=0.25.0 sentencepiece tensorboard
%pip install -q -e {PROJECT_DIR}

In [ ]:
# Cell 4 - 確認 GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# Cell 5 - 訓練參數設定（依需求調整）
import sys
sys.path.insert(0, PROJECT_DIR)

BATCH_SIZE   = 16      # A100 40GB 建議 16~32
TOTAL_STEPS  = 100000
SAVE_ITER    = 10000
NUM_EXPERTS  = 8
TOP_K        = 2
LR           = 5e-4
MIXED_PREC   = "bf16" # A100 支援 bf16，若用 T4 改成 fp16

print("Training config:")
print(f"  batch_size  = {BATCH_SIZE}")
print(f"  total_steps = {TOTAL_STEPS}")
print(f"  num_experts = {NUM_EXPERTS}, top_k = {TOP_K}")
print(f"  output_dir  = {OUTPUT_DIR}")

In [ ]:
# Cell 6 - 開始訓練
from mario_gpt import MarioDataset, MarioLM, TrainingConfig, MarioGPTTrainer

TOKENIZER_PATH = "shyamsn97/Mario-GPT2-700-context-length"

mario_lm = MarioLM(
    lm_path="random",
    tokenizer_path=TOKENIZER_PATH,
    use_moe=True,
    num_experts=NUM_EXPERTS,
    moe_top_k=TOP_K,
)

dataset = MarioDataset(mario_lm.tokenizer)

config = TrainingConfig(
    output_dir=OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    total_steps=TOTAL_STEPS,
    save_iteration=SAVE_ITER,
    aux_loss_coeff=0.01,
    learning_rate=LR,
    mixed_precision=MIXED_PREC,
)

trainer = MarioGPTTrainer(mario_lm, dataset, config=config)
trainer.train(TOTAL_STEPS)